# Week 6 — Spark Data Processing Assignment
**Objective:** Understand Spark architecture and perform efficient data processing using
transformations, filtering, schema handling, and optimized file formats (CSV vs Parquet).

**Dataset:** `data/source.csv` — a small retail/e-commerce style product dataset
(product_id, product_name, category, price, quantity, region, priority, status).

**Pipeline:** Read → Schema Validation → Clean Nulls → Filter → Transform → Write (CSV & Parquet)


## 1. Spark Session Creation
Create a `SparkSession` — the entry point to any Spark application. The **Driver** runs this code and creates a logical plan; the **Cluster Manager** (here, Spark's built-in *local* manager) allocates resources; **Executors** run the actual tasks.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, round as spark_round, upper
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

spark = SparkSession.builder \
    .appName("Week6_Spark_Data_Processing") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("Spark Version         :", spark.version)
print("Application Name      :", spark.sparkContext.appName)
print("Master                :", spark.sparkContext.master)
print("Default Parallelism   :", spark.sparkContext.defaultParallelism)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/26 21:55:38 WARN Utils: Your hostname, vm, resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/07/26 21:55:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/usr/local/lib/python3.12/dist-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/07/26 21:55:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Version         : 4.2.0
Application Name      : Week6_Spark_Data_Processing
Master                : local[*]
Default Parallelism   : 1


## 2. Load Dataset
Read the CSV with an **explicit schema** (best practice for large/production datasets — avoids the extra read pass that `inferSchema=True` triggers) and, separately, with `inferSchema=True` as required by the assignment.

In [2]:
explicit_schema = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("region", StringType(), True),
    StructField("priority", StringType(), True),
    StructField("status", StringType(), True),
])

df = spark.read.csv("../data/source.csv", header=True, schema=explicit_schema)

df_inferred = spark.read.csv("../data/source.csv", header=True, inferSchema=True)

print("Row count:", df.count())
print("Columns  :", df.columns)

Row count: 20
Columns  : ['product_id', 'product_name', 'category', 'price', 'quantity', 'region', 'priority', 'status']


## 3. Schema Inspection

In [3]:
df.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- status: string (nullable = true)



In [4]:
df_inferred.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- status: string (nullable = true)



## 4. Data Cleaning — Handle Null Values
Check how many nulls exist in critical columns, then drop rows with a missing `price` (a required business field) and fill missing `quantity` with `0` instead of discarding those rows.

In [5]:
null_counts = df.select([
    (col(c).isNull().cast("int")).alias(c) for c in ["price", "quantity"]
]).groupBy().sum().collect()[0]

print("Null count in 'price'   :", null_counts['sum(price)'])
print("Null count in 'quantity':", null_counts['sum(quantity)'])

Null count in 'price'   : 2
Null count in 'quantity': 3


In [6]:
df_clean = df.dropna(subset=["price"]).fillna({"quantity": 0})
print("Row count after cleaning:", df_clean.count())

Row count after cleaning: 18


## 5. Filtering Rows
Select only `Electronics` category products.

In [7]:
electronics_df = df_clean.filter(col("category") == "Electronics")
electronics_df.show(truncate=False)

+----------+-----------------+-----------+-------+--------+------+--------+---------+
|product_id|product_name     |category   |price  |quantity|region|priority|status   |
+----------+-----------------+-----------+-------+--------+------+--------+---------+
|101       |Wireless Mouse   |Electronics|799.5  |120     |North |Low     |Completed|
|102       |Bluetooth Speaker|Electronics|1999.0 |45      |South |Medium  |Completed|
|105       |LED Monitor      |Electronics|8999.0 |25      |North |Medium  |Completed|
|108       |Smartphone       |Electronics|15999.0|60      |West  |High    |Completed|
|109       |Gaming Keyboard  |Electronics|2499.0 |0       |North |Medium  |Pending  |
|112       |Laptop Stand     |Electronics|899.0  |80      |West  |Medium  |Completed|
|114       |Desk Lamp        |Electronics|650.0  |90      |South |Low     |Completed|
|116       |Tablet           |Electronics|12999.0|35      |North |High    |Completed|
|119       |Router           |Electronics|1799.0 |0   

## 6. Column Selection

In [8]:
selected_df = electronics_df.select("product_id", "price")
selected_df.show(truncate=False)

+----------+-------+
|product_id|price  |
+----------+-------+
|101       |799.5  |
|102       |1999.0 |
|105       |8999.0 |
|108       |15999.0|
|109       |2499.0 |
|112       |899.0  |
|114       |650.0  |
|116       |12999.0|
|119       |1799.0 |
+----------+-------+



## 7. Rename Columns

In [9]:
renamed_df = df_clean.withColumnRenamed("product_name", "item_name") \
                      .withColumnRenamed("price", "unit_price")
print(renamed_df.columns)

['product_id', 'item_name', 'category', 'unit_price', 'quantity', 'region', 'priority', 'status']


## 8. Data Type Casting

In [10]:
casted_df = renamed_df.withColumn("unit_price", col("unit_price").cast(DoubleType())) \
                       .withColumn("product_id", col("product_id").cast(StringType()))
casted_df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- item_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- quantity: integer (nullable = false)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- status: string (nullable = true)



## 9. Add New Columns
Add `final_price` (price + 18% tax) and an uppercase `category_upper` column.

In [11]:
final_df = renamed_df.withColumn("final_price", spark_round(col("unit_price") * 1.18, 2)) \
                      .withColumn("category_upper", upper(col("category")))
final_df.show(truncate=False)

+----------+-----------------+-----------+----------+--------+------+--------+---------+-----------+--------------+
|product_id|item_name        |category   |unit_price|quantity|region|priority|status   |final_price|category_upper|
+----------+-----------------+-----------+----------+--------+------+--------+---------+-----------+--------------+
|101       |Wireless Mouse   |Electronics|799.5     |120     |North |Low     |Completed|943.41     |ELECTRONICS   |
|102       |Bluetooth Speaker|Electronics|1999.0    |45      |South |Medium  |Completed|2358.82    |ELECTRONICS   |
|103       |Office Chair     |Furniture  |4500.0    |15      |East  |High    |Pending  |5310.0     |FURNITURE     |
|104       |Notebook Set     |Stationery |150.0     |300     |West  |Low     |Completed|177.0      |STATIONERY    |
|105       |LED Monitor      |Electronics|8999.0    |25      |North |Medium  |Completed|10618.82   |ELECTRONICS   |
|106       |Study Table      |Furniture  |3200.0    |10      |South |Low

## 10. Handle Null Values (final safety net before writing output)

In [12]:
final_df = final_df.na.fill({"quantity": 0})
print("Nulls remaining in quantity:",
      final_df.filter(col("quantity").isNull()).count())

Nulls remaining in quantity: 0


## 11. Transformations — Wide Transformation Example (Shuffle)
`groupBy().agg()` is a **wide transformation**: it requires data with the same key to be brought together across partitions, which triggers a **shuffle** (an expensive network + disk operation). Compare this to **narrow transformations** like `filter`/`select`/`withColumn`, where each output partition depends on only one input partition (no shuffle needed).

In [13]:
agg_df = final_df.groupBy("category") \
                  .agg({"final_price": "avg", "quantity": "sum"}) \
                  .withColumnRenamed("avg(final_price)", "avg_final_price") \
                  .withColumnRenamed("sum(quantity)", "total_quantity")
agg_df.show(truncate=False)

+-----------+--------------+------------------+
|category   |total_quantity|avg_final_price   |
+-----------+--------------+------------------+
|Electronics|455           |6115.349999999999 |
|Furniture  |115           |3527.728          |
|Stationery |670           |462.85499999999996|
+-----------+--------------+------------------+



## 12. Actions
`count()`, `show()`, and `collect()` are **actions** — they trigger execution of the whole lineage (DAG) built up so far. Everything before this point (`filter`, `select`, `withColumn`, `groupBy`) was **lazy** and only recorded as a logical plan until an action was called.

In [14]:
row_count_action = final_df.count()   # ACTION — triggers execution
print("Total rows:", row_count_action)

Total rows: 18


### 12b. Filter — AND condition
Rows where `status == 'Completed'` **AND** `final_price > 1000`.

In [15]:
high_value_completed = final_df.filter((col("status") == "Completed") & (col("final_price") > 1000))
high_value_completed.show(truncate=False)

+----------+-----------------+-----------+----------+--------+------+--------+---------+-----------+--------------+
|product_id|item_name        |category   |unit_price|quantity|region|priority|status   |final_price|category_upper|
+----------+-----------------+-----------+----------+--------+------+--------+---------+-----------+--------------+
|102       |Bluetooth Speaker|Electronics|1999.0    |45      |South |Medium  |Completed|2358.82    |ELECTRONICS   |
|105       |LED Monitor      |Electronics|8999.0    |25      |North |Medium  |Completed|10618.82   |ELECTRONICS   |
|108       |Smartphone       |Electronics|15999.0   |60      |West  |High    |Completed|18878.82   |ELECTRONICS   |
|110       |Bookshelf        |Furniture  |2750.0    |18      |South |Low     |Completed|3245.0     |FURNITURE     |
|112       |Laptop Stand     |Electronics|899.0     |80      |West  |Medium  |Completed|1060.82    |ELECTRONICS   |
|116       |Tablet           |Electronics|12999.0   |35      |North |Hig

### 12c. Filter — OR condition
Rows where `region == 'North'` **OR** `priority == 'High'`.

In [16]:
priority_or_region = final_df.filter((col("region") == "North") | (col("priority") == "High"))
priority_or_region.show(truncate=False)

+----------+---------------+-----------+----------+--------+------+--------+---------+-----------+--------------+
|product_id|item_name      |category   |unit_price|quantity|region|priority|status   |final_price|category_upper|
+----------+---------------+-----------+----------+--------+------+--------+---------+-----------+--------------+
|101       |Wireless Mouse |Electronics|799.5     |120     |North |Low     |Completed|943.41     |ELECTRONICS   |
|103       |Office Chair   |Furniture  |4500.0    |15      |East  |High    |Pending  |5310.0     |FURNITURE     |
|105       |LED Monitor    |Electronics|8999.0    |25      |North |Medium  |Completed|10618.82   |ELECTRONICS   |
|108       |Smartphone     |Electronics|15999.0   |60      |West  |High    |Completed|18878.82   |ELECTRONICS   |
|109       |Gaming Keyboard|Electronics|2499.0    |0       |North |Medium  |Pending  |2948.82    |ELECTRONICS   |
|116       |Tablet         |Electronics|12999.0   |35      |North |High    |Completed|15

## 13. Save as CSV

In [17]:
final_df.coalesce(1).write.mode("overwrite").option("header", True).csv("../output/processed_csv")
print("CSV written to ../output/processed_csv")

CSV written to ../output/processed_csv


## 14. Save as Parquet

In [18]:
final_df.write.mode("overwrite").parquet("../output/processed_parquet")
print("Parquet written to ../output/processed_parquet")

Parquet written to ../output/processed_parquet


## 15. Performance Explanation — CSV vs Parquet + Predicate Pushdown
Compare on-disk size, then read back the Parquet output with a filter and inspect the **physical plan** to confirm **Predicate Pushdown** — Spark pushes the filter condition down into the Parquet reader itself (`PushedFilters`), so unmatched row groups are skipped and never even loaded into memory.

In [19]:
import os

def dir_size(path):
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            total += os.path.getsize(os.path.join(dirpath, f))
    return total

csv_size = dir_size("../output/processed_csv")
parquet_size = dir_size("../output/processed_parquet")

print("CSV output size (bytes)     :", csv_size)
print("Parquet output size (bytes) :", parquet_size)

CSV output size (bytes)     : 1499
Parquet output size (bytes) : 3830


In [20]:
parquet_check = spark.read.parquet("../output/processed_parquet").filter(col("final_price") > 1000)
parquet_check.explain(True)

== Parsed Logical Plan ==
'Filter '`>`('final_price, 1000)
+- Relation [product_id#331,item_name#332,category#333,unit_price#334,quantity#335,region#336,priority#337,status#338,final_price#339,category_upper#340] parquet

== Analyzed Logical Plan ==
product_id: int, item_name: string, category: string, unit_price: double, quantity: int, region: string, priority: string, status: string, final_price: double, category_upper: string
Filter (final_price#339 > cast(1000 as double))
+- Relation [product_id#331,item_name#332,category#333,unit_price#334,quantity#335,region#336,priority#337,status#338,final_price#339,category_upper#340] parquet

== Optimized Logical Plan ==
Filter (isnotnull(final_price#339) AND (final_price#339 > 1000.0))
+- Relation [product_id#331,item_name#332,category#333,unit_price#334,quantity#335,region#336,priority#337,status#338,final_price#339,category_upper#340] parquet

== Physical Plan ==
*(1) Filter (isnotnull(final_price#339) AND (final_price#339 > 1000.0))
+- *(

### Key takeaway
Look for **`PushedFilters: [IsNotNull(final_price), GreaterThan(final_price,1000.0)]`** in the
physical plan above — that's Spark telling the Parquet reader to skip data at the file/row-group
level before it ever reaches memory, which is not possible with row-based CSV.

## Best Practices Followed
- Used `.show()` for inspection, **never `.collect()`**, on the full dataset (avoids pulling all
  rows into driver memory — critical on multi-GB/TB datasets).
- Used an **explicit schema** on the primary read path to avoid the extra scan `inferSchema` costs.
- Cleaned nulls *before* transformations, so downstream aggregations aren't skewed by missing data.
- Used `coalesce(1)` only for the CSV output (small demo dataset) — on real large datasets this
  would be avoided or replaced with `repartition` tuned to cluster size, since forcing everything
  into 1 file removes parallelism.
- Wrote the final output in **Parquet** (columnar, compressed, supports predicate pushdown) as the
  preferred format for downstream analytics.


In [21]:
spark.stop()
print("Spark session stopped.")

Spark session stopped.
